# Macro Data Exploration

Download and explore economy data from Polygon/Massive REST API.

Endpoints:
- `/fed/v1/treasury-yields` — daily yield curve (1Y, 5Y, 10Y)
- `/fed/v1/inflation` — monthly CPI
- `/fed/v1/inflation-expectations` — monthly model-based expectations
- `/fed/v1/labor-market` — monthly unemployment + participation

In [1]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv('../.env')
API_KEY = os.getenv('POLYGON_S3_SECRET_KEY')
BASE    = 'https://api.polygon.io'

def fetch_all(path, params=None):
    """Fetch all pages from a paginated endpoint."""
    params  = params or {}
    headers = {'Authorization': f'Bearer {API_KEY}'}
    results = []
    url     = f'{BASE}{path}'
    params['limit'] = 50000
    while url:
        r = requests.get(url, headers=headers, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        results.extend(data.get('results', []))
        url    = data.get('next_url')
        params = {}  # cursor already embedded in next_url
        print(f'  fetched {len(results)} rows...', end='\r')
    print()
    return results

print('Ready. API key loaded:', bool(API_KEY))

Ready. API key loaded: True


## 1. Treasury Yields

In [2]:
print('Fetching treasury yields...')
raw = fetch_all('/fed/v1/treasury-yields')
df_yields = pd.DataFrame(raw)
df_yields['date'] = pd.to_datetime(df_yields['date'])
df_yields = df_yields.set_index('date').sort_index()

print(f'Shape: {df_yields.shape}')
print(f'Date range: {df_yields.index.min()} to {df_yields.index.max()}')
print(f'Columns: {df_yields.columns.tolist()}')
print()
print(df_yields.tail(10))

Fetching treasury yields...
  fetched 16047 rows...
Shape: (16047, 7)
Date range: 1962-01-02 00:00:00 to 2026-04-02 00:00:00
Columns: ['yield_1_year', 'yield_5_year', 'yield_10_year', 'yield_2_year', 'yield_30_year', 'yield_3_month', 'yield_1_month']

            yield_1_year  yield_5_year  yield_10_year  yield_2_year  \
date                                                                  
2026-03-20          3.80          4.01           4.39          3.88   
2026-03-23          3.76          3.95           4.34          3.83   
2026-03-24          3.81          4.03           4.39          3.90   
2026-03-25          3.77          3.96           4.33          3.84   
2026-03-26          3.83          4.08           4.42          3.96   
2026-03-27          3.77          4.06           4.44          3.88   
2026-03-30          3.71          3.97           4.35          3.82   
2026-03-31          3.68          3.92           4.30          3.79   
2026-04-01          3.68          3.97

In [3]:
# Compute derived features
df_yields['spread_10y_1y'] = df_yields['yield_10_year'] - df_yields['yield_1_year']  # curve slope
df_yields['spread_5y_1y']  = df_yields['yield_5_year']  - df_yields['yield_1_year']
df_yields['yield_10y_mom_20d'] = df_yields['yield_10_year'].diff(20)   # 1-month momentum
df_yields['yield_10y_mom_60d'] = df_yields['yield_10_year'].diff(60)   # 3-month momentum
df_yields['curve_mom_20d']     = df_yields['spread_10y_1y'].diff(20)

print('Last 5 rows with derived features:')
print(df_yields[['yield_1_year','yield_5_year','yield_10_year',
                  'spread_10y_1y','yield_10y_mom_20d','curve_mom_20d']].tail(5))

Last 5 rows with derived features:
            yield_1_year  yield_5_year  yield_10_year  spread_10y_1y  \
date                                                                   
2026-03-27          3.77          4.06           4.44           0.67   
2026-03-30          3.71          3.97           4.35           0.64   
2026-03-31          3.68          3.92           4.30           0.62   
2026-04-01          3.68          3.97           4.33           0.65   
2026-04-02          3.68          3.94           4.31           0.63   

            yield_10y_mom_20d  curve_mom_20d  
date                                          
2026-03-27               0.47           0.18  
2026-03-30               0.30           0.13  
2026-03-31               0.24           0.11  
2026-04-01               0.24           0.14  
2026-04-02               0.18           0.09  


## 2. Inflation

In [4]:
print('Fetching inflation...')
raw = fetch_all('/fed/v1/inflation')
df_cpi = pd.DataFrame(raw)
df_cpi['date'] = pd.to_datetime(df_cpi['date'])
df_cpi = df_cpi.set_index('date').sort_index()

print(f'Shape: {df_cpi.shape}')
print(f'Date range: {df_cpi.index.min()} to {df_cpi.index.max()}')
print(f'Columns: {df_cpi.columns.tolist()}')
print()
print(df_cpi.tail(10))

Fetching inflation...
  fetched 950 rows...
Shape: (950, 6)
Date range: 1947-01-01 00:00:00 to 2026-02-01 00:00:00
Columns: ['cpi', 'cpi_year_over_year', 'cpi_core', 'pce', 'pce_core', 'pce_spending']

                cpi  cpi_year_over_year  cpi_core      pce  pce_core  \
date                                                                   
2025-05-01  320.620                 NaN   326.893  126.380   125.790   
2025-06-01  321.435                 NaN   327.658  126.743   126.121   
2025-07-01  322.169                 NaN   328.682  126.960   126.430   
2025-08-01  323.291                 NaN   329.700  127.293   126.714   
2025-09-01  324.245                 NaN   330.418  127.625   126.954   
2025-10-01      NaN                 NaN       NaN  127.873   127.245   
2025-11-01  325.063                 NaN   331.043  128.155   127.473   
2025-12-01  326.031                 NaN   331.814  128.615   127.929   
2026-01-01  326.588                 NaN   332.793  128.969   128.394   
2026-0

## 3. Inflation Expectations

In [5]:
print('Fetching inflation expectations...')
raw = fetch_all('/fed/v1/inflation-expectations')
df_exp = pd.DataFrame(raw)
df_exp['date'] = pd.to_datetime(df_exp['date'])
df_exp = df_exp.set_index('date').sort_index()

print(f'Shape: {df_exp.shape}')
print(f'Date range: {df_exp.index.min()} to {df_exp.index.max()}')
print(f'Columns: {df_exp.columns.tolist()}')
print()
print(df_exp.tail(10))

Fetching inflation expectations...
  fetched 531 rows...
Shape: (531, 7)
Date range: 1982-01-01 00:00:00 to 2026-03-01 00:00:00
Columns: ['model_1_year', 'model_5_year', 'model_10_year', 'model_30_year', 'market_5_year', 'market_10_year', 'forward_years_5_to_10']

            model_1_year  model_5_year  model_10_year  model_30_year  \
date                                                                   
2025-06-01      2.380542      2.337052       2.346975       2.468291   
2025-07-01      2.789047      2.372164       2.338801       2.449353   
2025-08-01      2.689360      2.304636       2.278358       2.412305   
2025-09-01      2.803599      2.336189       2.298842       2.422122   
2025-10-01      2.727598      2.321751       2.296421       2.424534   
2025-11-01      2.703875      2.330943       2.307130       2.431894   
2025-12-01      3.192869      2.420234       2.347133       2.442364   
2026-01-01      2.586395      2.337320       2.331881       2.453990   
2026-02-01     

## 4. Labor Market

In [6]:
print('Fetching labor market...')
raw = fetch_all('/fed/v1/labor-market')
df_labor = pd.DataFrame(raw)
df_labor['date'] = pd.to_datetime(df_labor['date'])
df_labor = df_labor.set_index('date').sort_index()

print(f'Shape: {df_labor.shape}')
print(f'Date range: {df_labor.index.min()} to {df_labor.index.max()}')
print(f'Columns: {df_labor.columns.tolist()}')
print()
print(df_labor.tail(10))

Fetching labor market...
  fetched 939 rows...
Shape: (939, 4)
Date range: 1948-01-01 00:00:00 to 2026-03-01 00:00:00
Columns: ['unemployment_rate', 'labor_force_participation_rate', 'job_openings', 'avg_hourly_earnings']

            unemployment_rate  labor_force_participation_rate  job_openings  \
date                                                                          
2025-06-01                4.1                            62.3        7204.0   
2025-07-01                4.3                            62.2        7089.0   
2025-08-01                4.3                            62.3        6919.0   
2025-09-01                4.4                            62.5        7169.0   
2025-10-01                NaN                             NaN        7170.0   
2025-11-01                4.5                            62.5        6846.0   
2025-12-01                4.4                            62.4        6550.0   
2026-01-01                4.3                            62.1     

## 5. Combine into USD Macro Feature Set

In [7]:
# Resample everything to daily, forward-fill (monthly data fills each day)
daily_idx = pd.date_range('2009-01-01', '2025-12-31', freq='D')

macro = pd.DataFrame(index=daily_idx)

# Yields — already daily
for col in ['yield_1_year', 'yield_5_year', 'yield_10_year',
            'spread_10y_1y', 'yield_10y_mom_20d', 'yield_10y_mom_60d', 'curve_mom_20d']:
    if col in df_yields.columns:
        macro[col] = df_yields[col].reindex(daily_idx).ffill()

# CPI — monthly, forward fill
for col in df_cpi.columns:
    macro[f'cpi_{col}'] = df_cpi[col].reindex(daily_idx).ffill()

# CPI momentum
if 'cpi_cpi' in macro.columns:
    macro['cpi_yoy']   = macro['cpi_cpi'].pct_change(365) * 100
    macro['cpi_mom3m'] = macro['cpi_cpi'].pct_change(90)  * 100

# Inflation expectations — monthly
for col in df_exp.columns:
    macro[f'exp_{col}'] = df_exp[col].reindex(daily_idx).ffill()

# Real rate = 10Y yield - 1Y inflation expectation
if 'yield_10_year' in macro.columns and 'exp_model_1_year' in macro.columns:
    macro['real_rate_10y'] = macro['yield_10_year'] - macro['exp_model_1_year']

# Labor — monthly
for col in df_labor.columns:
    macro[f'labor_{col}'] = df_labor[col].reindex(daily_idx).ffill()

# Unemployment trend
if 'labor_unemployment_rate' in macro.columns:
    macro['unemp_mom3m'] = macro['labor_unemployment_rate'].diff(90)

macro = macro.dropna(how='all')
print(f'Macro feature set: {macro.shape}')
print(f'Date range: {macro.index.min()} to {macro.index.max()}')
print(f'\nColumns:')
for c in macro.columns:
    print(f'  {c:<35} last={macro[c].dropna().iloc[-1]:.4f}')

Macro feature set: (6209, 28)
Date range: 2009-01-01 00:00:00 to 2025-12-31 00:00:00

Columns:
  yield_1_year                        last=3.4800
  yield_5_year                        last=3.7300
  yield_10_year                       last=4.1800
  spread_10y_1y                       last=0.7000
  yield_10y_mom_20d                   last=0.0900
  yield_10y_mom_60d                   last=0.0800
  curve_mom_20d                       last=0.2000
  cpi_cpi                             last=326.0310
  cpi_cpi_year_over_year              last=2.3113
  cpi_cpi_core                        last=331.8140
  cpi_pce                             last=128.6150
  cpi_pce_core                        last=127.9290
  cpi_pce_spending                    last=21455.5000
  cpi_yoy                             last=2.6533
  cpi_mom3m                           last=0.5508
  exp_model_1_year                    last=3.1929
  exp_model_5_year                    last=2.4202
  exp_model_10_year                   last=

## 6. Quick Correlation with EURUSD Direction

In [8]:
# Does any macro feature correlate with EURUSD 4H forward direction?
df_1h = pd.read_parquet('../backend/data/processed/EURUSD_1H.parquet')
df_1h = df_1h[df_1h.index <= '2024-06-30']

c = df_1h['close']
fwd_4h = np.log(c.shift(-4) / c)

# Align macro to 1H bars (daily macro -> each 1H bar gets that day's macro value)
macro_1h = macro.reindex(df_1h.index.normalize()).set_index(df_1h.index)

print(f'{"Feature":<35} {"Corr_4H":>10}')
print('=' * 48)
results = []
for col in macro.columns:
    x   = macro_1h[col]
    valid = pd.DataFrame({'x': x, 'fwd': fwd_4h}).dropna()
    if len(valid) < 1000: continue
    corr = np.corrcoef(valid['x'].values, valid['fwd'].values)[0, 1]
    results.append((col, corr))

for col, corr in sorted(results, key=lambda x: abs(x[1]), reverse=True):
    flag = ' ***' if abs(corr) > 0.02 else (' *' if abs(corr) > 0.01 else '')
    print(f'{col:<35} {corr:>+10.4f}{flag}')

Feature                                Corr_4H
yield_10y_mom_20d                      -0.0100 *
cpi_cpi_year_over_year                 -0.0071
spread_10y_1y                          -0.0070
exp_market_10_year                     -0.0068
cpi_yoy                                -0.0066
exp_market_5_year                      -0.0064
exp_model_1_year                       -0.0062
exp_forward_years_5_to_10              -0.0053
exp_model_30_year                      -0.0046
labor_labor_force_participation_rate    -0.0041
exp_model_10_year                      -0.0040
exp_model_5_year                       -0.0040
yield_1_year                           +0.0035
yield_10_year                          -0.0031
labor_avg_hourly_earnings              +0.0029
cpi_cpi_core                           +0.0026
yield_5_year                           -0.0025
cpi_pce_core                           +0.0024
cpi_pce                                +0.0019
cpi_cpi                                +0.0019
cpi_pce_sp